In [15]:
from unittest import result

from debugpy.launcher import output
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-opus-4-1"

In [16]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [17]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)



In [18]:
# Call the function to execute it
dataset = generate_dataset()
dataset

[{'task': 'Write a Python function that takes an S3 bucket name and object key as parameters and returns a pre-signed URL valid for 1 hour using boto3',
  'format': 'python'},
 {'task': "Create a JSON policy document that allows read-only access to all objects in an S3 bucket named 'my-data-bucket' for any AWS principal",
  'format': 'json'},
 {'task': "Write a regex pattern that validates AWS ARN format and captures the service name and resource id from ARNs like 'arn:aws:ec2:us-west-2:123456789012:instance/i-1234567890abcdef0'",
  'format': 'regex'}]

In [19]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=4)

In [20]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then return the rsult"""
    prompt=f"""
    Please solve the following task:

    {test_case['task']}

    *Respond only with Python, JSON, or a plain Regex.
    *Do not add any comments or commentary or explanations
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [21]:
def run_test_case(test_case):
    """Calls run_prompt and then grades the result"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return{
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [22]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

NameError: name 'grade_by_model' is not defined

In [ ]:
print(json.dumps(results, indent=2))